# 📬 UC2: Nachrichten-Triage
### Knowledge Navigator 1987 → 2026

**Originalszene:** Phil spielt dem Professor drei Nachrichten vor und gibt einen
priorisierten Überblick — Forschungsteam aus Guatemala, Student Jordan, seine Mutter.

**Heute:** Statt einer Sprachnachricht analysiert eine KI automatisch deine E-Mails
und sortiert sie in Sekunden nach Priorität und Handlungsbedarf.

---
**Lernziele:**
- Claude API aufrufen
- CO-STAR Prompt-Framework anwenden
- Strukturierten JSON-Output verarbeiten
- ipywidgets für interaktive Notebooks nutzen

**Tech Stack:** Python · Anthropic Claude API · ipywidgets · python-dotenv

In [ ]:
# ── Setup: Bibliotheken laden & API-Key ─────────────────────────────────────
# Für lokale Entwicklung: API-Key aus .env Datei
# Für Deepnote: Environment Variable im Projekt-Panel setzen
# REGEL: Niemals den API-Key direkt in den Code schreiben!

import json
import os
from pathlib import Path

import anthropic
import ipywidgets as widgets
from dotenv import load_dotenv
from IPython.display import display, HTML

# .env laden (lokal) — auf Deepnote wird die Umgebungsvariable direkt genutzt
load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
if not ANTHROPIC_API_KEY:
    raise ValueError(
        "ANTHROPIC_API_KEY nicht gefunden!\n"
        "Lokal: .env Datei anlegen (siehe .env.example)\n"
        "Deepnote: Environment Variables im Projekt-Panel setzen"
    )

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("✅ Claude-Client initialisiert")

In [ ]:
# ── CO-STAR Prompt Template ──────────────────────────────────────────────────
#
# Das CO-STAR-Framework strukturiert LLM-Prompts in 6 Dimensionen:
#
#   C — Context:    Wer ist der Assistent? In welchem Kontext arbeitet er?
#   O — Objective:  Was soll konkret erreicht werden?
#   S — Style:      Wie soll die Antwort formuliert sein?
#   T — Tone:       Welcher Ton ist angemessen?
#   A — Audience:   Für wen ist die Ausgabe bestimmt?
#   R — Response:   Welches Format soll die Antwort haben?
#
# Zusätzlich: Chain-of-Thought — die Kategorien sind explizit definiert,
# damit Claude nachvollziehbar klassifiziert.
# ─────────────────────────────────────────────────────────────────────────────

COSTAR_PROMPT = """\
C (Context): Du bist ein intelligenter E-Mail-Assistent für einen Hochschuldozenten.
Du hilfst dabei, eingehende E-Mails schnell zu priorisieren.

O (Objective): Analysiere die folgende E-Mail. Bestimme Kategorie, Priorität,
erstelle eine Kurzzusammenfassung und empfehle eine konkrete Aktion.

S (Style): Strukturiert, präzise, ohne Füllwörter.

T (Tone): Professionell und sachlich.

A (Audience): Der Dozent möchte in 5 Sekunden entscheiden,
welche Mails sofortige Aufmerksamkeit brauchen.

R (Response): Antworte AUSSCHLIESSLICH mit validem JSON — kein Text davor oder danach:
{{
    \"kategorie\": \"VIP\" | \"Aktion nötig\" | \"Nur Info\" | \"Ignorieren\",
    \"priorität\": 1 | 2 | 3 | 4,
    \"zusammenfassung\": \"Max. 2 prägnante Sätze.\",
    \"empfohlene_aktion\": \"Konkrete, sofort umsetzbare Empfehlung.\"
}}

Kategorien:
- VIP          (Priorität 1): Dekanat, Vorgesetzte, wichtige Partner
- Aktion nötig (Priorität 2): Studierende, Kollegen mit konkreten Anfragen
- Nur Info     (Priorität 3): Newsletter, FYI-Mails, Informationen ohne Handlungsbedarf
- Ignorieren   (Priorität 4): Spam, Werbung, irrelevant

E-Mail:
{email_text}
"""

print("✅ CO-STAR Prompt Template geladen")

In [ ]:
# ── Kernfunktion: analyze_email ──────────────────────────────────────────────

def analyze_email(email_text: str) -> dict:
    """
    Analysiert eine E-Mail per Claude API und gibt eine strukturierte Triage zurück.

    Args:
        email_text: Vollständiger E-Mail-Text (Betreff + Absender + Body)

    Returns:
        dict: {kategorie, priorität, zusammenfassung, empfohlene_aktion}
    """
    prompt = COSTAR_PROMPT.format(email_text=email_text)

    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )

    raw = response.content[0].text.strip()
    return json.loads(raw)


def format_result_html(result: dict, email_preview: str) -> str:
    """Formatiert das Triage-Ergebnis als farbiges HTML."""
    farben = {
        "VIP":          ("#dc2626", "🔴"),
        "Aktion nötig": ("#d97706", "🟡"),
        "Nur Info":     ("#2563eb", "🔵"),
        "Ignorieren":   ("#6b7280", "⚫"),
    }
    farbe, emoji = farben.get(result["kategorie"], ("#6b7280", "⚫"))
    preview = email_preview[:80].replace('<', '&lt;').replace('>', '&gt;')

    return f"""
    <div style="border-left: 4px solid {farbe}; padding: 12px 16px;
                margin: 8px 0; background: #f9fafb; border-radius: 4px;">
        <div style="font-size: 18px; font-weight: bold; color: {farbe};">
            {emoji} {result['kategorie']} &nbsp;
            <span style="font-size:13px; color:#6b7280;">
                Priorität {result['priorität']}/4
            </span>
        </div>
        <div style="margin-top: 8px; color: #374151;">
            <b>Zusammenfassung:</b> {result['zusammenfassung']}
        </div>
        <div style="margin-top: 4px; color: #374151;">
            <b>Empfehlung:</b> {result['empfohlene_aktion']}
        </div>
        <div style="margin-top: 8px; font-size: 11px; color: #9ca3af;">
            E-Mail-Vorschau: {preview}...
        </div>
    </div>
    """

print("✅ Funktionen definiert")

In [ ]:
# ── Interaktives UI mit ipywidgets ───────────────────────────────────────────

email_input = widgets.Textarea(
    placeholder="E-Mail hier einfügen (Von: / Betreff: / Text) ...",
    layout=widgets.Layout(width="100%", height="200px")
)

analyse_btn = widgets.Button(
    description="📬 Analysieren",
    button_style="primary",
    layout=widgets.Layout(width="160px", height="40px")
)

output = widgets.Output()

def on_analyse_click(b):
    with output:
        output.clear_output()
        email_text = email_input.value.strip()
        if not email_text:
            display(HTML("<p style='color:red'>⚠️ Bitte E-Mail-Text einfügen.</p>"))
            return
        display(HTML("<p>⏳ Analysiere...</p>"))
        try:
            result = analyze_email(email_text)
            output.clear_output()
            display(HTML(format_result_html(result, email_text)))
            print("\n📄 Rohes JSON (für Entwickler):")
            print(json.dumps(result, ensure_ascii=False, indent=2))
        except json.JSONDecodeError as e:
            display(HTML(f"<p style='color:red'>❌ JSON-Fehler: {e}</p>"))
        except Exception as e:
            display(HTML(f"<p style='color:red'>❌ Fehler: {e}</p>"))

analyse_btn.on_click(on_analyse_click)

display(
    widgets.HTML("<h3>📬 E-Mail einfügen und analysieren</h3>"),
    email_input,
    analyse_btn,
    output
)

In [ ]:
# ── Demo: Alle 4 Sample-E-Mails auf einmal analysieren ───────────────────────

sample_dir = Path("sample_emails")
sample_files = sorted(sample_dir.glob("*.txt"))

print(f"🔍 Analysiere {len(sample_files)} Sample-E-Mails...\n")

results_html = "<h3>📊 Triage-Ergebnisse</h3>"
for f in sample_files:
    email_text = f.read_text(encoding="utf-8")
    result = analyze_email(email_text)
    results_html += format_result_html(result, email_text)

display(HTML(results_html))

## 🚀 Erweiterungen (Stufe 2)

### exchangelib-Anbindung
Statt E-Mails manuell einzufügen, verbindet sich das Notebook mit deinem
Exchange-Postfach (THWS oder DHBW) und analysiert automatisch alle ungelesenen Mails.

```python
from exchangelib import Credentials, Account

credentials = Credentials(username="vorname.nachname@thws.de", password="***")
account = Account("vorname.nachname@thws.de", credentials=credentials, autodiscover=True)

for mail in account.inbox.filter(is_read=False).order_by("-datetime_received")[:20]:
    result = analyze_email(f"Von: {mail.sender}\nBetreff: {mail.subject}\n\n{mail.text_body}")
    print(result)
```

### Batch-Analyse + Export
Ergebnisse als CSV oder HTML-Report exportieren.

### Feedback-Loop
Benutzer korrigiert die Kategorie → wird für Few-Shot-Prompts oder Fine-Tuning gespeichert.